# Run the optical pipeline in Google Colab
Use a **CPU runtime**. Run cells in order. Default: 96 ONTs × 90 days × five-minute
samples. This notebook clones Branch 2 and runs the complete development pipeline.
No GPU is used. Runtime/memory availability varies; use the small configuration
only to check installation before the full run. Final assessment stays unopened.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path("/content/anomaly_detection")
if not REPO.exists():
    subprocess.run([
        "git", "clone", "--branch", "codex/branch-2", "--single-branch",
        "https://github.com/LiliDopidze/anomaly_detection.git", str(REPO),
    ], check=True)
%cd /content/anomaly_detection
%pip install -q -e ".[dev]"


## Choose where outputs will be saved
With `SAVE_TO_DRIVE=True`, authorise the Google Drive mount when prompted. All
outputs go directly into your chosen MyDrive run folder. With False, outputs stay
under `/content` and can disappear when the runtime is recycled. Use a new RUN_NAME
for changed settings/code. Mounting Drive requires your own Google authorisation.

In [ ]:
import yaml

SAVE_TO_DRIVE = True
RUN_NAME = "optical_v7_run01"
SMALL_SMOKE_RUN = False

settings = yaml.safe_load((REPO / "configs/config.yaml").read_text())
if SMALL_SMOKE_RUN:
    settings["generator"].update(entities=8, days=14)
    RUN_NAME += "_smoke"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    output = Path("/content/drive/MyDrive/anomaly_detection") / RUN_NAME
else:
    output = REPO / "outputs" / RUN_NAME
settings["output"] = str(output)
CONFIG_PATH = Path("/content/optical_run.yaml")
CONFIG_PATH.write_text(yaml.safe_dump(settings))
print("All run outputs:", output)


In [ ]:
import json
import pandas as pd
from optical_anomaly.pipeline import prepare, develop

RUN = prepare(CONFIG_PATH)
print("Generated/reused measurements:", RUN / "telemetry.parquet")
print("Topology:", RUN / "topology.parquet")
display(pd.read_parquet(RUN / "topology.parquet").head())
display(pd.Series(json.loads((RUN / "generation_checks.json").read_text())["checks"]))
if not (RUN / "model.joblib").exists():
    develop(CONFIG_PATH)
comparison = pd.read_csv(RUN / "validation_comparison.csv")
display(comparison[["detector", "opening_intervals", "closing_intervals",
                    "detected", "faults", "pre_impact_recall",
                    "nuisance_per_1000_entity_days", "meets_workload_budget"]])


In [ ]:
import matplotlib.pyplot as plt

cutoff = pd.Timestamp(json.loads((RUN / "manifest.json").read_text())["split_boundaries"][0])
example = pd.read_parquet(
    RUN / "telemetry.parquet",
    filters=[("time", "<", cutoff), ("device", "==", "ONT-000")],
    columns=["time", "rx_dbm", "upstream_rx_dbm", "ont_temperature_c"],
).set_index("time")
example[["rx_dbm", "upstream_rx_dbm"]].plot(figsize=(12, 3), ylabel="dBm")
plt.show()
display(pd.read_csv(RUN / "validation_faults.csv").head())
print("Saved outputs:")
for path in sorted(RUN.iterdir()):
    print(path.name)


## Optional final assessment
Keep the next flag False while developing. Opening final assessment is a one-time
operation using frozen code, data and model. A new folder does not make previously
inspected data unseen. See RUN_GUIDE.md for downloading outputs and the five more
detailed local notebooks. Synthetic results do not prove operator performance.

In [ ]:
OPEN_FINAL_TEST = False
if OPEN_FINAL_TEST:
    from optical_anomaly.pipeline import final_evaluation
    display(pd.Series(final_evaluation(RUN)))
